# Incremental Delaunay triangulation

Runs [incremental.py](incremental.py) on the same four point sets as
[divide_and_conquer.ipynb](divide_and_conquer.ipynb). Points are inserted one by one into a big virtual
triangle, whose incircle is the circumcircle of the points' bounding box times a scale. Each point is located
by walking from a virtual corner, splits its triangle into three, and edges are flipped in a wave outwards. At
the end the virtual triangle is removed.

Each point set is drawn twice:

- **Scale 1**, the virtual triangle as first specified. Red marks the part of the convex hull no triangle
  covers. A thin triangle on a hull edge has a circumcircle that reaches far out past the edge: about
  `p·q / h`, for an apex at height `h` over an edge its foot splits into `p` and `q`. When that circle reaches
  a virtual corner, the triangle loses its edge to one with the corner and goes with it. A bigger triangle
  only makes this rarer: no finite size is safe for every point set.
- **Corners at infinity.** Corner `k` is `centre + λ u_k` as `λ` grows without bound. Every test involving a
  corner is a polynomial in `λ` whose sign is that of its highest nonzero coefficient. The circle through a
  hull edge and a corner at infinity is the half-plane beyond the edge, which never holds a point, so no
  triangle is lost and the boundary is the convex hull.

In each picture, the left side shows the triangulation just before the virtual corners are removed (edges to
corners at infinity are drawn as parallel rays), the middle the result, and the right side its dual, the
Voronoi diagram, with a dotted ray outwards past every boundary edge. With triangles missing, the rays
follow the dented boundary instead of the convex hull.

In [ ]:
import numpy as np

from incremental import IncrementalTriangulation, draw, virtual_size

SEED = 1
SCALES = 1, np.inf  # The virtual triangle's incircle over the bounding box's circumcircle

def triangulate_and_draw(points, title):
    for scale in SCALES:
        t = IncrementalTriangulation(points, scale)
        print(f'{virtual_size(scale)}: {len(t.triangles())} triangles, boundary of {len(t.hull)} '
              f'({"convex" if t.hull and t.is_convex() else "not convex"}), {t.flips} flips, '
              f'{t.steps} walk steps, {len(t.lost)} points lost, {len(t.illegal_edges())} illegal edges left')
        draw(t, f'{title}, {virtual_size(scale)}')

## 27 points, uniform in the unit square

The same points as `python incremental.py` with its default seed.

In [ ]:
triangulate_and_draw(np.random.default_rng(SEED).uniform(size=(27, 2)),
                     f'27 uniform random points (seed {SEED})')

## 225 points, uniform in the unit square

In [ ]:
triangulate_and_draw(np.random.default_rng(SEED).uniform(size=(225, 2)),
                     f'225 uniform random points (seed {SEED})')

## A donut: points uniform between two circles

Uniform by area: the radius is the square root of a uniform pick between the squared radii.

In [ ]:
def donut(n, inner, outer, seed):
    rng = np.random.default_rng(seed)
    radius = np.sqrt(rng.uniform(inner ** 2, outer ** 2, n))
    angle = rng.uniform(0, 2 * np.pi, n)
    return np.c_[radius * np.cos(angle), radius * np.sin(angle)]

triangulate_and_draw(donut(300, inner=0.5, outer=1.0, seed=SEED),
                     f'300 random points between circles of radius 0.5 and 1 (seed {SEED})')

## A circle and a semicircle

100 points evenly on a circle, and 100 on a smaller semicircle with a flat bottom, shifted up and to the right.
All the points of each shape lie on one circle, so their Delaunay triangulation is not unique, and this one
can differ from the divide and conquer one while both are correct. The red still only marks what is missing.
The long tangents along the hull are the worst case for a finite virtual triangle: long edges, with the arcs
hugging them, make a run of thin triangles with huge circumcircles.

In [ ]:
def circle_and_semicircle(n=100):
    angle = 2 * np.pi * np.arange(n) / n
    circle = np.c_[np.cos(angle), np.sin(angle)]
    # Both ends of the semicircle included: they are the corners of its flat bottom
    angle = np.pi * np.arange(n) / (n - 1)
    semicircle = np.c_[2.2 + 0.8 * np.cos(angle), 0.9 + 0.8 * np.sin(angle)]
    return np.r_[circle, semicircle]

triangulate_and_draw(circle_and_semicircle(), '100 points on a circle and 100 on a semicircle')